# H&M Dataset

Go go: https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations, and download the `articles.csv`, `customers.csv`, and `transactions_train.csv` files.

The following notebook with create the train and test sets to use in the learning.

In [2]:
import pandas as pd
import numpy as np
import datetime
from tqdm import tqdm

In [3]:
#load dataset
df = pd.read_csv("transactions_train.csv", dtype={"article_id": str})
customer_df = pd.read_csv('customers.csv', dtype={"article_id": str,'product_code':str})
article_df = pd.read_csv("articles.csv", dtype={"article_id": str})
df['product_code'] = df['article_id'].map(article_df.set_index('article_id')['product_code'])

In [4]:
df["t_dat"] = pd.to_datetime(df["t_dat"])
df["week"] = (df["t_dat"].max() - df["t_dat"]).dt.days // 7
print(df["week"].value_counts())

week
65    635891
13    549443
64    518946
42    518403
12    517428
       ...  
38    208382
27    205868
26    198818
44    191375
41    176816
Name: count, Length: 105, dtype: int64


In [5]:
df.head()

,t_dat,customer_id,article_id,price,sales_channel_id,product_code,week
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.050831,2,663713,104
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.030492,2,541518,104
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0505221004,0.015237,2,505221,104
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687003,0.016932,2,685687,104
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687004,0.016932,2,685687,104


In [6]:
customer_df.head()

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...


In [7]:
article_df.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,0108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,0108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,0108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,0110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,0110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [8]:
#select top 6 product types
chosen_types = article_df.groupby(['product_type_name'])['product_code'].count().sort_values()[-6:].index
#select customers whose product type purchases exceeded 120
chosen_customers = df.groupby(['customer_id'])['product_code'].nunique()[df.groupby(['customer_id'])['product_code'].nunique() > 120].index

In [9]:
#top 5000 locations
location_customers = customer_df['postal_code'].value_counts()[1:5001].index
customer_df = customer_df.loc[customer_df['postal_code'].isin(location_customers)]

In [10]:
chosen_articles = article_df[article_df.product_type_name.isin(chosen_types)]
df = df[df['product_code'].isin(chosen_articles.product_code)]
df = df[df['customer_id'].isin(chosen_customers)]
df = df[df['customer_id'].isin(customer_df.customer_id)]

In [11]:
df.drop(['t_dat', 'price','article_id','sales_channel_id'], inplace=True, axis=1)
df['bought'] = 1
df.reset_index(inplace=True)

In [12]:
def create_dataset(df, week):
    hist_df = df[df["week"] > week]
    val_df = df[df["week"] <= week]

    return hist_df,val_df

train_df,val_df = create_dataset(df,50)
train_df.shape, val_df.shape

((121167, 5), (95223, 5))

In [13]:
train_df = train_df.groupby(['customer_id','product_code'])['bought'].count().reset_index()
val_df = val_df.groupby(['customer_id','product_code'])['bought'].count().reset_index()

In [14]:
#only consider those customers and items existing before.
frequent_users = train_df.customer_id.values
val_df = val_df[val_df['customer_id'].isin(frequent_users)]
frequent_items = train_df.product_code.values
val_df = val_df[val_df['product_code'].isin(frequent_items)]

In [15]:
n_users = train_df['customer_id'].nunique()
n_items = train_df['product_code'].nunique()

print("The number of customer: ",n_users)
print("The number of article: ",n_items)
print(train_df.shape)
print(val_df.shape)

The number of customer:  1760
The number of article:  8618
(86480, 3)
(30253, 3)


In [16]:
#encoder
customer_id2index = {c: i for i, c in enumerate(train_df['customer_id'].unique())}
item_id2index = {a: i for i, a in enumerate(train_df['product_code'].unique())}

train_df.customer_id = train_df.customer_id.map(customer_id2index)
train_df.product_code = train_df.product_code.map(item_id2index)

val_df.customer_id = val_df.customer_id.map(customer_id2index)
val_df.product_code = val_df.product_code.map(item_id2index)

In [17]:
train_df

,customer_id,product_code,bought
0,0,0,2
1,0,1,2
2,0,2,1
3,0,3,1
4,0,4,1
...,...,...,...
86475,1759,408,1
86476,1759,1052,2
86477,1759,1411,1
86478,1759,906,1


In [18]:
val_df

,customer_id,product_code,bought
0,0,1389,1
1,0,3240,1
2,0,2767,1
3,0,846,1
4,0,411,1
...,...,...,...
67683,1759,322,1
67686,1759,1811,1
67688,1759,2214,2
67690,1759,2283,1


In [19]:
train_df.to_csv("hm_train_df.csv", index=False)

In [20]:
val_df.to_csv("hm_val_df.csv", index=False)

In [21]:
# The number of customer:  1760
# The number of article:  8618

In [22]:
# FedAvg data setup(cluster data to each user)
# split_train = dict()
# for category in range(1760):
#     split_train[category] = train_df[train_df['customer_id'] == category]